# Phase 01

### 단계 01에 대한 코드를 정리한 것입니다.
1. Data Collection and Preprocessing
2. Purpose Extraction
3. LLM-Tagging & Split the multi purpose review data by LLM
4. CR Extraction

In [8]:
# ==========================================
# 1. Standard Library & Basic Imports
# ==========================================
import os
import sys
import re
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import gensim
import gensim.corpora as corpora
import praw
import google.generativeai as genai
import ast

# ==========================================
# 2. From Imports (Specific Modules)
# ==========================================
# 기초 및 유틸리티
from datetime import datetime
from tqdm.auto import tqdm
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

# AI 및 API 클라이언트
from openai import OpenAI

# Scikit-learn (머신러닝 및 텍스트 처리)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation

# Gensim (토픽 모델링 및 분석)
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from gensim.models import CoherenceModel

# BERTopic (토픽 모델링)
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# KeyBERT (키워드 추출)
from keybert import KeyBERT

# ==========================================
# 3. Initialization
# ==========================================
load_dotenv()

True

In [9]:
# ==========================================
# 4. Environment Setting
# ==========================================

# Reddit
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
user_agent = os.getenv("USER_AGENT")

reddit = praw.Reddit(
    client_id = client_id,
    client_secret = client_secret,
    user_agent = user_agent
)

# OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key = OPENAI_API_KEY)

In [10]:
# ==========================================
# 5. Function Definitions
# ==========================================

# NLP preprocessing 함수 정의 (목적 신호 강화 버전)
# Spacy 모델 로드 (품사 태깅 및 표제어 추출용)
# 설치 필요: python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

# 1. 기술 속성 구조화 함수 (기존 로직 유지)
def disambiguate_specs(text):
    text = text.lower()
    # 가격, 프레임, 해상도, 전력 등 수치 데이터의 의미론적 통합
    text = re.sub(r'\$(\d+)', r'price_\1', text)
    text = re.sub(r'\b(\d+)\s?(dollars|usd|bucks)\b', r'price_\1', text)
    text = re.sub(r'\b(\d{2,3})\s?(fps|hz)\b', r'fps_\1', text)
    text = re.sub(r'\b(\d{3,4})[pP]\b', r'res_\1p', text)
    text = re.sub(r'\b4[kK]\b', r'res_4k', text)
    text = re.sub(r'\b(\d+)\s?w\b', r'power_\1w', text)
    return text

# 2. 행위 중심(Activity-centric) 텍스트 정제 함수
def purpose_driven_clean(text):
    if not isinstance(text, str): return ""
    
    # A. 기본 노이즈 제거 및 스펙 단위 변환
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.replace('\n', ' ')
    text = disambiguate_specs(text)
    text = re.sub(r'[^a-z0-9_]', ' ', text) # 언더바(_) 보존
    
    # B. SpaCy를 이용한 정밀 분석
    doc = nlp(text)
    tokens = []
    
    # 목적 도출에 핵심적인 품사들만 선택 (명사, 동사, 형용사)
    # VERB를 포함함으로써 'playing', 'editing', 'working' 등 행위 정보를 포착
    allowed_postags = ['NOUN', 'VERB', 'ADJ']
    
    # 연구에서 '목적'을 정의하기 위해 반드시 살아있어야 하는 단어들 (기존 삭제 항목에서 제외)
    # 'game', 'play' 등이 살아있어야 'Gaming' 목적 군집이 형성됩니다.
    keep_for_purpose = {'game', 'play', 'use', 'work', 'study', 'edit', 'code', 'run'}
    
    for token in doc:
        # 불용어(Stopwords)이더라도 목적 관련 핵심어라면 통과
        is_purpose_keyword = token.text in keep_for_purpose
        
        if (is_purpose_keyword) or (not token.is_stop and token.pos_ in allowed_postags):
            # 표제어 추출 (Lemmatization)
            lemma = token.lemma_
            
            # 너무 짧은 단어 필터링 (숫자나 구조화된 스펙 제외)
            if len(lemma) > 2 or lemma.isdigit() or "_" in lemma:
                tokens.append(lemma)
    
    return " ".join(tokens)

# 3. 데이터프레임 적용 파이프라인
def preprocess_for_purpose_discovery(df):
    print("전처리 시작: 사용 목적(Purpose) 신호 강화 모드")
    
    # 원문이 삭제된 경우 제외
    df = df[~df['Body'].isin(['[deleted]', '[removed]'])].copy()
    
    # 전처리 적용 (tqdm으로 진행률 표시)
    tqdm.pandas()
    df['cleaned_Body'] = df['Body'].progress_apply(purpose_driven_clean)
    
    # 단어 수 기반 필터링 (정보량이 너무 적은 리뷰 제외)
    df['word_count'] = df['cleaned_Body'].apply(lambda x: len(str(x).split()))
    df = df[df['word_count'] >= 5] 
    
    print(f"전처리 완료. 최종 데이터 수: {len(df)}")
    return df

# LLM-Tagging 함수 정의
def get_openai_tags(review_text):
    # 시스템 프롬프트: 역할과 출력 형식을 강력하게 정의
    system_instruction = """
        ### Role: Senior NPD Research Data Auditor

        Your sole mission is to determine if a review is "Actionable Product Feedback"
        and classify its usage purpose.

        ## Step 1. Validity Check (is_valid):
        Do NOT look for noise patterns. Instead, ask yourself:

        "Can I complete this sentence from this review?
        'A product designer should [do X] because this user needs [Y].'"

        - If YES → proceed to Step 2
        - If NO  → is_valid: false, primary_purpose: "None", stop here

        A valid review must satisfy ALL of the following:
        1. It is a self-contained statement (not a reply fragment or conversation).
        2. It expresses a concrete preference, expectation, or complaint about
        the product's design, performance, or ergonomics.
        3. A product designer can extract a concrete requirement from it.
        The primary intent is NOT to seek help, ask for recommendations,
        or get answers from other users.

        ## Step 2. Evidence Collection (BEFORE assigning purpose):
        List the specific signals in the review that indicate Professional or Casual intent.

        - You MUST find at least 2 concrete, distinct signals to assign a purpose.
        - If you cannot find 2 signals, set primary_purpose: "None", is_valid: false.

        Signals for **Professional**:
            The laptop is used as a tool for specialized, skill-based work
            where performance directly impacts output quality or productivity.
            (e.g., software dev, data science, 4K editing, 3D rendering,
            heavy corporate multitasking, engineering tools, etc.)

        Signals for **Casual**:
            The laptop is used primarily for consumption, communication,
            or light tasks without domain-specific performance demands.
            (e.g., web browsing, streaming, social media, 
            basic student homework, everyday home use, etc.)

        ## Step 3. Confidence Check:
        Ask yourself: "If I read this review 10 times, would I assign 
        the same purpose every time with >90% certainty?"

        - If YES → assign the purpose
        - If NO, or if Professional and Casual signals are mixed/equal → 
        primary_purpose: "None", is_valid: false

        ## Output Schema (JSON Only):
        Important: Fill in "purpose_signals" BEFORE deciding "primary_purpose".
        {
            "purpose_signals": ["signal 1 from the review", "signal 2 from the review"],
            "is_valid": boolean,
            "primary_purpose": "Professional" | "Casual" | "None",
            "designer_action": "Complete the sentence: 'A designer should [X] because this user needs [Y].' Leave empty string if is_valid is false.",
            "reasoning": "One sentence explaining the validity and purpose decision.",
            "evidence_snippet": "The most relevant quote. Empty string if invalid."
        }
    """

    user_prompt = f"""
        Review: "{review_text}"""
    
    try:
        response = client.chat.completions.create(
            model = "gpt-5.1",
            service_tier = "flex", # gpt 5.1의 flex를 사용하는 것을 명시적으로 반영
            messages = [
                {"role" : "system", "content" : system_instruction},
                {"role" : "user", "content" : user_prompt}
            ],
            temperature = 0,
            response_format = {'type' : 'json_object'}
        )

        return response.choices[0].message.content
    except Exception as e:
        print(f"Error : {e}")
        return None

# 파싱
def parse_purpose(text): # purpose_classification 컬럼의 값을 파싱하여 리스트로 반환
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return re.findall(r"'([^']*)'", text)

# LLM-분해 함수 정의
def decompose_review(review_text, purpose):
    # 시스템 프롬프트 정의
    system_instruction = """
        ### Role: Linguistic Analyst & Expert in User Research

        Your goal is to perform "Review Atomization." You will receive a laptop review that contains multiple intents and its pre-assigned categories. Your task is to decompose the single review into multiple independent segments, ensuring each segment corresponds to exactly one "Usage Purpose."

        ### Purpose Categories:

        1. Academic/Education: Focuses on school-related tasks, college life, note-taking, online learning, research, and general student productivity.
        2. Gaming/Hobby: Focuses on high-performance gaming, e-sports, hobbyist modding, and entertainment that requires high GPU/CPU performance.
        3. Professional: Focuses on "Work-critical" tasks including software development, data science, video editing, 3D rendering, graphic design, and corporate-level multi-tasking.
        4. Everyday/Casual: Focuses on light daily use, media consumption (Netflix/YouTube), social media, and web browsing without a specific professional or academic goal.
        5. General/Unspecified: Hardware-centric reviews that discuss build quality, battery, or screen WITHOUT mentioning a specific usage context.

        ### Task Instructions:

        1. Decompose by Intent: Split the input review into separate "Atomic Units." Each unit must contain only ONE usage purpose.
        2. Contextual Integrity: Each decomposed segment must be a self-contained sentence. If the original text uses pronouns (e.g., "it," "this laptop"), replace them with the actual subject to ensure the segment is understandable on its own.
        3. 1:1 Mapping: Ensure that each segment in the "review" list matches the corresponding category in the "purpose" list at the same index.
        4. Exclusivity: Do not leave out any meaningful part of the original review.

        ### Core Operational Rules:

        1. **Extraction, Not Generation**: Do not rephrase or repeat the same preamble (e.g., "I am looking for...") in every segment. Extract the unique requirement for each purpose directly from the text.
        2. **Strict Purpose Mapping**: The number of output segments in the "review" list must match the number of categories in the "purpose" list. Do not create extra segments for fillers.
        3. **Handle Noise/General Talk**: If a part of the review is just a general question or a filler (e.g., "I don't know what to buy"), merge it into the most relevant purpose segment or discard it if it lacks specific requirements.
        4. **Context Recovery (Minimal)**: Ensure each segment is a grammatically complete sentence. Use neutral subjects like "This laptop" or "The device" only when necessary to replace pronouns (it, this). Do not over-edit.
        5. **Length Control**: The total information density of the output segments must be equivalent to the original text. Avoid inflating the dataset with redundant words.

        ### Output Schema (JSON only):

        {
        "review": ["string"]
        "purpose": ["string"]
        }

        ### Example:

        Input:
        - Review: "The screen is vibrant and perfect for my evening Netflix sessions, but the fan gets quite loud when I'm compiling heavy code for work."
        - Purposes: ["Everyday/Casual", "Professional"]

        Output:
        {
        "review": [
            "The screen is vibrant and perfect for my evening Netflix sessions.",
            "The fan gets quite loud when I am compiling heavy code for work."
        ],
        "purpose": [
            "Everyday/Casual",
            "Professional"
        ]
        }
    """
    
    user_prompt = f"""
        Review: {review_text}
        Purposes: {purpose}
    """

    try:
        response = client.chat.completions.create(
            model = "gpt-5.1",
            service_tier = "flex",
            messages = [
                {"role" : "system", "content" : system_instruction},
                {"role" : "user", "content" : user_prompt}
            ], temperature = 0,
            response_format = {"type" : "json_object"}
        )

        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None

# CR Extraction 함수 정의
# 목적별 Customer Requirement 추출용 프롬프트 함수
def get_cr_extraction_prompt(purpose):
    return f"""
        ### Identity: The Voice of the "{purpose}" User Group
        You are the representative of the "{purpose}" community. You understand their lifestyle, daily challenges, and the specific performance standards they demand from a laptop. 

        ### Mission:
        Read the following review segments and extract **Atomic Customer Requirements (CR)**. 
        A "CR" must represent a single, distinct need of the user. Your goal is to provide concise, standardized labels that can be directly mapped to engineering characteristics in a QFD matrix.

        ### CR Extraction Rules (Strict):
        1. **Atomic & Single-Intent**: Each requirement must contain only ONE specific need. If a review mentions weight and battery, split them into two separate CRs. 
        2. **Noun-Phrase Format**: Use short, punchy noun phrases. Avoid full sentences like "The laptop should have..." or "It must be...".
        3. **Exclude Reasons & Context**: Do not include "why" or "how" in the requirement field (e.g., No "for portability" or "so that it doesn't throttle"). Put all supporting context in the 'evidence_summary'. 
        4. **Objective Translation**: Convert subjective feedback into objective standards where possible.

        ### Examples:
        - ❌ Bad: "Lightweight chassis under 1.5kg for better portability during commute." (Too long, includes reason)
        - ✅ Good: "Body weight (under 1.5kg)"
        - ❌ Bad: "High-resolution screen with good color for photo editing." (Includes reason)
        - ✅ Good: "Color accuracy (DCI-P3 100%)"

        ### Output Schema (JSON only):
        {{
            "representative_persona": "Briefly describe your identity as a {purpose} user",
            "requirements": [
                {{
                    "requirement": "The atomic, noun-phrase label (e.g., 'Battery life', 'Fan noise')",
                    "evidence_summary": "Short snippet of what users actually said"
                }}
            ]
        }}
    """

# Customer Requirement 추출 함수 정의
def run_cr_extraction(purpose, review_segments, client):
    system_prompt = get_cr_extraction_prompt(purpose)
    joined_reviews = "\n".join(review_segments)
    user_prompt = f"""
        Target Purpose : "{purpose}"
        Review Segments : \n{joined_reviews}
    """
    
    try:
        response = client.chat.completions.create(
            model = "gpt-5.1",
            service_tier = "flex",
            messages = [
                {"role" : "system", "content" : system_prompt},
                {"role" : "user", "content" : user_prompt}
            ], temperature = 0,
            response_format = {"type" : "json_object"}
        )
        
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error during LLM call for purpose '{purpose}': {e}")
        return None

### 1. Data Collection and Preprocessing

In [7]:
# ==========================================
# 1. Data Collection
# ==========================================

# 전체 게시글 수 카운팅
# 기준은 1000개가 넘는지에 대해서

search_results = reddit.subreddit("laptops").hot(limit = 1000)

# 리스트로 변환하여 개수 확인
post_count = len(list(search_results))

print(f"--------------------------------------------------")
print(f"📊 검색 결과 분석")
print(f"📍 Subreddit : r/laptops")
print(f"📝 Total Posts : {post_count}개")
print(f"--------------------------------------------------") 

if post_count < 100:
    print("⚠️ 데이터가 너무 적습니다. 모델 변경이나 키워드 확장을 권장합니다.")
elif post_count >= 900:
    print("✅ 데이터가 충분하거나 API 한계치에 도달했습니다. 충분한 실험이 가능합니다.")

--------------------------------------------------
📊 검색 결과 분석
📍 Subreddit : r/laptops
📝 Total Posts : 999개
--------------------------------------------------
✅ 데이터가 충분하거나 API 한계치에 도달했습니다. 충분한 실험이 가능합니다.


In [5]:
# 데이터 수집 설정
target_subreddit = "laptops"
"""
post_limit = 1000 - 2026.02.19
post_limit = 3000 - 2026.02.23
"""
post_limit = 1000
review_lst = []

print(f"🤖 [Data Collection] Reddit 크롤링을 시작합니다...")
print(f"--------------------------------------------------")
print(f"📍 Target Subreddit : {target_subreddit}")
print(f"📊 Post Limit       : {post_limit}")
print(f"--------------------------------------------------")

search_results = reddit.subreddit(target_subreddit).hot(limit = post_limit)

for submission in tqdm(search_results, total = post_limit, desc = "Collecting data"):
    # 게시글 정보 수집
    
    submission.comments.replace_more(limit=None)
    time.sleep(1.5) # 1.5초의 정지 -> 너무 많으면 429 Error가 나오기 때문

    # 제한 없이 모든 댓글 수집
    for comment in submission.comments.list():
        comment_info = {
            "Type": "Comment",
            "Body": comment.body,
            "Date": datetime.fromtimestamp(comment.created_utc).strftime('%Y-%m-%d %H:%M:%S'),
            "URL": f"https://www.reddit.com{comment.permalink}"
        }
        review_lst.append(comment_info)

df = pd.DataFrame(review_lst)
print(f"\n✅ 수집 완료! 총 {df.shape[0]}개의 행이 저장되었습니다.")

# df.to_excel('./reddit_reviews_02.xlsx') # 5543개행의 데이터 -> post_limit = 1000

🤖 [Data Collection] Reddit 크롤링을 시작합니다...
--------------------------------------------------
📍 Target Subreddit : laptops
📊 Post Limit       : 1000
--------------------------------------------------


KeyboardInterrupt: 

In [5]:
# 데이터 수집 설정
target_subreddit = "laptops"
# 각 카테고리별로 가져올 제한 (합쳐서 약 3000개를 목표로 함)
category_limit = 1000 
review_lst = []
seen_submission_ids = set()  # 중복 수집 방지를 위한 세트

print(f"🤖 [Data Collection] Reddit 크롤링을 시작합니다 (다중 정렬 모드)...")
print(f"--------------------------------------------------")
print(f"📍 Target Subreddit : {target_subreddit}")
print(f"📊 Target Categories: Hot, New, Top (Each {category_limit})")
print(f"--------------------------------------------------")

# 1. 수집할 제너레이터 리스트 생성
# 'top'의 경우 'all'(전체 기간)로 설정해야 더 많은 고유 데이터를 얻을 수 있습니다.
generators = [
    reddit.subreddit(target_subreddit).hot(limit=category_limit),
    reddit.subreddit(target_subreddit).new(limit=category_limit),
    reddit.subreddit(target_subreddit).top(limit=category_limit, time_filter='all')
]

for gen in generators:
    # 각 카테고리별로 tqdm 진행 (전체 흐름 파악용)
    for submission in tqdm(gen, total=category_limit, desc=f"Collecting from {gen.__class__.__name__}"):
        
        # 중복 체크: 이미 처리한 ID라면 건너뜀
        if submission.id in seen_submission_ids:
            continue
        seen_submission_ids.add(submission.id)

        try:
            # 게시글 정보 및 댓글 수집 로직
            submission.comments.replace_more(limit=None)
            time.sleep(1.2) # API Rate Limit을 고려한 대기 (약간 조정)

            for comment in submission.comments.list():
                comment_info = {
                    "Post_ID": submission.id,
                    "Type": "Comment",
                    "Body": comment.body,
                    "Date": datetime.fromtimestamp(comment.created_utc).strftime('%Y-%m-%d %H:%M:%S'),
                    "URL": f"https://www.reddit.com{comment.permalink}"
                }
                review_lst.append(comment_info)
                
            # 목표치(3000개 포스트)에 도달하면 중단하고 싶을 경우 아래 주석 해제
            # if len(seen_submission_ids) >= 3000:
            #     break

        except Exception as e:
            print(f"⚠️ Error processing submission {submission.id}: {e}")
            continue

df = pd.DataFrame(review_lst)
print(f"\n✅ 수집 완료!")
print(f"📌 고유 게시글 수: {len(seen_submission_ids)}개")
print(f"📌 총 댓글 행 수: {df.shape[0]}개")

# 결과 저장
# df.to_excel(f'./reddit_reviews_{datetime.now().strftime("%m%d_%H%M")}.xlsx', index=False)

🤖 [Data Collection] Reddit 크롤링을 시작합니다 (다중 정렬 모드)...
--------------------------------------------------
📍 Target Subreddit : laptops
📊 Target Categories: Hot, New, Top (Each 1000)
--------------------------------------------------



✅ 수집 완료!
📌 고유 게시글 수: 1505개
📌 총 댓글 행 수: 81276개


In [16]:
# 데이터가 저장되어있기 때문에 여기서 파일을 불러와서 위의 수집 과정은 현재 생략 (2026.02.23)

df = pd.read_excel("../Data/raw_data/reddit_reviews_03.xlsx")

df = df.loc[:10000]

In [7]:
# ==========================================
# 2. Preprocessing
# ==========================================

print(f"정제 전 : {df.shape[0]}")
df = preprocess_for_purpose_discovery(df) # 함수 호출
print(f"정제 후 : {df.shape[0]}")

정제 전 : 5343
전처리 시작: 사용 목적(Purpose) 신호 강화 모드


100%|██████████| 5320/5320 [00:16<00:00, 325.23it/s]

전처리 완료. 최종 데이터 수: 3361
정제 후 : 3361


### 2. Topic Modeling - Purpose Extraction

In [ ]:
# 데이터 준비

# Topic Modeling using LDA

print(f"Valid 전처리 이전 데이터 개수 : {df.shape[0]}")

_df = df.copy()

_df_valid = _df.dropna(subset = ['cleaned_Body']) # 결측치 제거
_df_valid = _df_valid[_df_valid['cleaned_Body'].str.strip().astype(bool)] # 빈 문자열 제거

print(f"Valid 전처리 이후 데이터 개수 : {_df_valid.shape[0]}")

In [ ]:
# ==========================================
# 1. Topic Modeling - LDA
# ==========================================

# 정의

vectorizer = CountVectorizer(
    stop_words = 'english',
    max_features = 1000,
    ngram_range = (1, 2),
    max_df = 0.9,
    min_df = 2,
) # 기본 구성

# Topic Modeling (K = 50)

K = 50 # 토픽 개수

_df = _df_valid.reset_index(drop = True).copy()
dtm = vectorizer.fit_transform(_df['cleaned_Body'])
feature_names = vectorizer.get_feature_names_out()

lda = LatentDirichletAllocation(
    n_components = K,
    random_state = 42,
    max_iter = 10,
    learning_method = 'online',
    n_jobs = -1,
)
lda.fit(dtm) # 학습

# 토픽 데이터 저장

topics = []
for topic_idx, topic in enumerate(lda.components_):
    top_feature_ind = topic.argsort()[-10:][::-1]
    top_features = [feature_names[i] for i in top_feature_ind]
    topics.append(', '.join(top_features))

topic_df = pd.DataFrame(
    [f"Topic : {i + 1}" for i in range(K)],
    columns = ['Topics']
)

topic_df['Topic_Doc'] = topics

# topic_df.to_excel('LDA_results.xlsx', index = False

In [ ]:
# ==========================================
# 2. Topic Modeling - BERTopic
# ==========================================

_df = _df_valid.reset_index(drop = True).copy()

docs = _df['cleaned_Body'].tolist()

# 모델 설정
umap_model = UMAP(n_neighbors = 50, n_components = 10, min_dist = 0.0, metric = 'cosine', random_state = 210)
hdbscan_model = HDBSCAN(min_cluster_size = 10, metric = 'euclidean', cluster_selection_method = 'eom', prediction_data = True)
vectorizer_model = CountVectorizer(
    stop_words = 'english',
    max_features = 1000,
    ngram_range = (1, 2),
    max_df = 0.9,
    min_df = 2
)

# 모델 학습
topic_model = BERTopic(
    embedding_model = "all-MiniLM-L6-v2",
    umap_model = umap_model,
    hdbscan_model = hdbscan_model,
    vectorizer_model = vectorizer_model,
    calculate_probabilities = False,
    verbose = True
)

topics, probs = topic_model.fit_transform(docs)

# 결과 추출

topic_data = []
info_df = topic_model.get_topic_info()

# 노이즈(-1) 제거
info_df = info_df[info_df['Topic'] != -1]

for index, row in info_df.iterrows():
    topic_id = row['Topic']
    
    keywords = topic_model.get_topic(topic_id)
    keyword_lst = [word for word, score in keywords][:10]
    keywords_str = ", ".join(keyword_lst)

    rep_docs = topic_model.get_representative_docs(topic_id)
    rep_docs_str = "\n--------------------\n".join(rep_docs)

    topic_data.append({
        "Topic": topic_id,
        "Count": row['Count'],
        "Top_Keywords": keywords_str,
        "Representative_Docs": rep_docs_str
    })

final_df = pd.DataFrame(topic_data)

# final_df.to_excel('./BERTopic_results.xlsx', index=False)

In [ ]:
# ==========================================
# 3. Key Extraction - KeyBERT
# ==========================================

_df = _df_valid.reset_index(drop = True).copy()

kb = KeyBERT()

docs = _df['cleaned_Body'].tolist()

keywords = kb.extract_keywords(
    docs,
    keyphrase_ngram_range = (1, 2),
    stop_words = 'english',
    use_maxsum = True,
    nr_candidates = 20,
    top_n = 5
)

keywords_lst = []
for i in range(len(keywords)):
    temp = []
    for j in range(len(keywords[i])):
        temp.append(keywords[i][j][0])
    temp_keywords = ", ".join(temp)
    keywords_lst.append(temp_keywords)

keywords_df = pd.DataFrame(
    [f"Keyword {i+1}" for i in range(len(keywords_lst))],
    columns = ['Keyword_Index']
)

keywords_df['Keywords'] = keywords_lst

# keywords_df.to_excel('./Keybert_results.xlsx', index = False)

### 3. LLM-Tagging & Split the multi purpose review data by LLM

In [ ]:
# ==========================================
# 1. LLM-Tagging
# ==========================================

# print("Tagging 시작... (사용자 성향 분석 중)")
# results = []

# for index, row in df.iterrows():
#     json_str = get_openai_tags(row['cleaned_Body'])
#     if json_str:
#         try:
#             data = json.loads(json_str)
#             results.append(data)
#         except json.JSONDecodeError:
#             results.append({"is_valid" : False, "error" : "JSON Parse Error"})
#     else:
#         results.append({"is_valid" : False, "error" : "No Response"})
    
#     time.sleep(1)

# df_tag = pd.DataFrame(results)
# df_tagged = pd.concat([df.reset_index(drop=True), df_tag], axis=1)

# df_tagged.to_excel('reddit_reviews_tagged.xlsx', index = False)

# 태깅 결과 확인
# for key in df_tagged[df_tagged['is_valid'] == True]['key_priority'].unique():
#     print(f"{key} percentage : {df_tagged[df_tagged['key_priority'] == key].shape[0] / df_tagged[df_tagged['is_valid'] == True].shape[0] * 100:.2f}%")

In [4]:
df = pd.read_excel('../Data/raw_data/reddit_reviews_03.xlsx')
df = df.loc[:5000]
df.shape

(5001, 5)

In [5]:
# ==========================================
# 1-1. LLM-Tagging
# ==========================================

print("Tagging 시작... (사용자 성향 분석 중)")

MAX_WORKERS = 10
results = [None] * len(df)

with ThreadPoolExecutor(max_workers = MAX_WORKERS) as executor:
    # 각 행의 인덱스와 텍스트를 함께 전달
    future_to_index = {
        executor.submit(get_openai_tags, row['Body']) : i
        for i, (idx, row) in enumerate(df.iterrows())
    }

    # 진행 바 표시 (tqdm)
    for future in tqdm(as_completed(future_to_index), total = len(df), desc = "Analyzing Reviews"):
        i = future_to_index[future]
        try:
            raw_result = future.result()
            results[i] = json.loads(raw_result)
        except json.JSONDecodeError: # JSON 형태가 아닐 경우
            results[i] = {"is_valid": False, "error": "JSON Parse Error", "purpose_classification": [], "reasoning": ""}
        except Exception as e: # 그 외의 에러
            results[i] = {"is_valid": False, "error": f"Thread Error: {e}", "purpose_classification": [], "reasoning": ""}

df_tag = pd.DataFrame(results)
df_tagged = pd.concat([df.reset_index(drop=True), df_tag], axis=1)

Tagging 시작... (사용자 성향 분석 중)


Analyzing Reviews:  97%|█████████▋| 4827/5001 [11:43<00:22,  7.83it/s]

Error : Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}


Analyzing Reviews: 100%|██████████| 5001/5001 [12:08<00:00,  6.87it/s]


In [6]:
df_tagged['is_valid'].value_counts()

is_valid
False    4398
True      603
Name: count, dtype: int64

In [7]:
df_tagged.to_excel('../Data/Preprocessed_data/reddit_reviews_tagged_enhanced_03-5.xlsx', index = False)

In [10]:
# ==========================================
# 2. Split the multi purpose review data by LLM
# ==========================================

df = df_tagged.copy()

df = df[df['is_valid'] == True] # 유효한 데이터만 가져오기

# df['purpose_classification'] = df['purpose_classification'].apply(parse_purpose)
print(f"Purpose_classification 목적 개수 : \n {df['purpose_classification'].apply(len).value_counts()}")

# 분해 과정 진행

print("🚀 Decompose review 시작...")
results = []

# 필터링

multi_purpose_review_idx = df[df['purpose_classification'].apply(lambda x: len(x) > 1)].index
multi_purpose_review = df.loc[multi_purpose_review_idx]
print(f"📊 전체 데이터 세트 크기 : {df.shape}")
print(f"🔍 다중 목적 리뷰(Multi-purpose) 크기 : {multi_purpose_review.shape}")

# LLM을 통한 작업 진행

for idx, row in multi_purpose_review.iterrows():
    print(f"📝 [{idx}] 리뷰 분석 중... 기존 목적 : {row['purpose_classification']}")
    _review = row['Body'] # 리뷰 데이터
    _purpose = row['purpose_classification'] # Purpose

    json_str = decompose_review(_review, _purpose)

    if json_str:
        try:
            data = json.loads(json_str)
            new_row = row.copy()
            new_row['Body'] = data['review']
            new_row['purpose_classification'] = data['purpose']

            results.append(new_row)
            print(f"  ✂️ 분해 완료 (Review): {data['review']}")
            print(f"  🏷️ 분해 완료 (Purpose): {data['purpose']}")
        except Exception as e:
            print(f"⚠️ JSON parsing error: {e}")
    else:
        print(f"❌ {idx} 리뷰 분해 실패")
    
    time.sleep(1)

temp_df = pd.DataFrame(results)
temp_df = temp_df.explode(['Body', 'purpose_classification'])
print(f"✨ 분해 및 변환 완료 데이터 : {temp_df.shape}")

df = df.drop(multi_purpose_review_idx)
df = pd.concat([df , temp_df])
print(f"✅ 최종 통합 데이터 결과 : {df.shape}")

df = df.reset_index(drop = True)

Purpose_classification 목적 개수 : 
 purpose_classification
1    2546
2     177
3      24
4       3
Name: count, dtype: int64
🚀 Decompose review 시작...
📊 전체 데이터 세트 크기 : (2750, 11)
🔍 다중 목적 리뷰(Multi-purpose) 크기 : (204, 11)
📝 [4] 리뷰 분석 중... 기존 목적 : ['Gaming/Hobby', 'Everyday/Casual']
  ✂️ 분해 완료 (Review): ['I’m looking for a laptop in the $500-$700 range that could work for playing games like League of Legends and the Sims 4, and I’ve looked at some gaming laptops but wasn’t sure what to go with or if I even need one.', 'I’m looking for a laptop in the $500-$700 range that could work for writing and light work.']
  🏷️ 분해 완료 (Purpose): ['Gaming/Hobby', 'Everyday/Casual']
📝 [7] 리뷰 분석 중... 기존 목적 : ['Gaming/Hobby', 'Professional']
  ✂️ 분해 완료 (Review): ['The laptop will be used for light gaming.', 'The laptop will eventually be used for some video editing.']
  🏷️ 분해 완료 (Purpose): ['Gaming/Hobby', 'Professional']
📝 [14] 리뷰 분석 중... 기존 목적 : ['Academic/Education', 'Professional']
  ✂️ 분해 완료 (Review): ['

In [11]:
for idx, val in enumerate(df['purpose_classification']):
    if isinstance(val, list):
        df.loc[idx, 'purpose_classification'] = val[0]

In [12]:
df.to_excel('../Data/Preprocessed_data/reddit_reviews_tagged_enhanced_decompose_03.xlsx', index = False)

### 4. CR Extraction

In [4]:
df = pd.read_excel("../Data/Preprocessed_data/reddit_reviews_tagged_enhanced_decompose_03.xlsx")

In [5]:
# Purpose_classification [] 제거

# for idx, val in enumerate(df['purpose_classification']):
#     if val.startswith('[') and val.endswith(']'):
#         val = val[2:-2]
#         df.loc[idx, 'purpose_classification'] = val

for idx, val in enumerate(df['purpose_classification']):
    if isinstance(val, list):
        df.loc[idx, 'purpose_classification'] = val[0]

result_df = pd.DataFrame()
for purpose in df['purpose_classification'].unique():
    if purpose == 'General/Unspecified': continue
    purpose_reviews = df[df['purpose_classification'] == purpose]
    cr_result = run_cr_extraction(
        purpose = purpose,
        review_segments = purpose_reviews['Body'].tolist(),
        client = client
    )
    
    if cr_result:
        try:
            cr_data = json.loads(cr_result)
            cr_df = pd.DataFrame(cr_data['requirements'])
            cr_df['purpose'] = purpose
            result_df = pd.concat([result_df, cr_df], ignore_index=True)
        except json.JSONDecodeError as e:
            print(f"JSON decoding error for purpose '{purpose}': {e}")
    time.sleep(2)  # API rate limit 고려

# result_df.to_excel('./all_purposes_cr_extraction_results_01.xlsx', index = False)

In [6]:
result_df.to_excel('../Data/CR_Extraction/all_purposes_cr_extraction_results_04.xlsx', index = False)